# Пример - Гриниан и дуальность полиформ

## Подготовка

Выполните все ячейки сверху вниз. Ревизия пакета: **e7a9647**.

In [1]:
from importlib.metadata import PackageNotFoundError, distribution
import json
import subprocess
import sys

source = "https://github.com/magin67/polyform_algebra/archive/e7a9647f448ca0ca743f89c067601e418ef604ed.zip"
try:
    installed = distribution("polyform_algebra")
    installed_url = json.loads(installed.read_text("direct_url.json") or "{}").get("url")
except PackageNotFoundError:
    installed = None
    installed_url = None

# A tag and a development revision can report the same version string.
if installed_url != source:
    if "polyform_algebra" in sys.modules:
        raise RuntimeError("Restart the kernel before installing the pinned package revision.")
    command = [sys.executable, "-m", "pip", "install", "-q"]
    if installed is not None and installed.version == "0.9.0":
        command += ["--force-reinstall", "--no-deps"]
    subprocess.check_call(command + [source])


In [2]:
import polyform_algebra
assert polyform_algebra.get_version() == "0.9.0"
print("polyform_algebra", polyform_algebra.get_version())

polyform_algebra 0.9.0


Девять расчётов проверяют экспоненты лапласиана и гриниана, формы разрезов, потенциалы и обращение коэффициентов.


Выполняйте фрагменты сверху вниз. Блокнот устанавливает закреплённую ревизию пакета. [polyform-algebra e7a9647](https://github.com/magin67/polyform_algebra/commit/e7a9647f448ca0ca743f89c067601e418ef604ed).


## Взаимная дуальность на взвешенном пути

Строим обе экспоненты и сравниваем все коэффициенты в обоих направлениях удаления. Для весов 2, 3 и 5 предельные коэффициенты равны 30 и $1/480$.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D0%BB%D0%B0%D0%BF%D0%BB%D0%B0%D1%81%D0%B8%D0%B0%D0%BD-%D0%B8-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD---%D0%B2%D0%B7%D0%B0%D0%B8%D0%BC%D0%BD%D0%B0%D1%8F-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C-%D1%8D%D0%BA%D1%81%D0%BF%D0%BE%D0%BD%D0%B5%D0%BD%D1%82#%D0%BF%D0%BE%D0%BB%D0%BD%D1%8B%D0%B9-%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80-%D0%BF%D1%83%D1%82%D1%8C-%D0%BD%D0%B0-%D1%87%D0%B5%D1%82%D1%8B%D1%80%D1%91%D1%85-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D0%B0%D1%85).

In [3]:
from collections import Counter
from fractions import Fraction
from itertools import combinations
from math import prod

from sympy import Matrix, expand, eye, symbols, zeros

from polyform_algebra.core import (
    equivalent, form_exp, grade_part, green, laplacian, laplacian_form,
    network, partition_form, potential, resistance, spanning_coefficient, spanning_tree_weight,
)
from polyform_algebra.duality import (
    affine_coefficients, cut_partition_form, dual_coefficients, greenian_form,
)
from polyform_algebra.kvp import signed_forest_toponome
from polyform_algebra.toponomes import laplacian_polynomial, vertex_toponome


def path(weights):
    return network(tuple("abcd"), tuple((a, b, w) for (a, b), w in zip(("ab", "bc", "cd"), weights)))


def exponential(net, constructor):
    return form_exp(constructor(net).value).value


weighted = path((2, 3, 5))
T = partition_form(weighted.context, (weighted.context.labels,)).value
M = exponential(weighted, laplacian_form)
N = exponential(weighted, greenian_form)
tau = spanning_coefficient(M).value
sigma = spanning_coefficient(N).value
assert (tau, sigma) == (30, Fraction(1, 480))
assert dual_coefficients(M, sigma * T) == affine_coefficients(N)
assert dual_coefficients(N, tau * T) == affine_coefficients(M)
print("Weighted P₄: tau = 30, sigma = 1/480, both deletions agree")

Weighted P₄: tau = 30, sigma = 1/480, both deletions agree


## Канонические векторные коэффициенты

Вычисляем знаковые коэффициенты из резистенсов двойным центрированием и сверяем их с внедиагональными элементами гриниана. Проверяем знаковый остовной вес.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D0%BA%D0%B0%D0%BD%D0%BE%D0%BD%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%BE%D0%B5-%D0%B2%D0%B5%D0%BA%D1%82%D0%BE%D1%80%D0%BD%D0%BE%D0%B5-%D0%BF%D1%80%D0%B5%D0%B4%D1%81%D1%82%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0#%D0%BF%D0%BE%D0%BB%D0%BD%D1%8B%D0%B9-%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80-%D0%BF%D1%83%D1%82%D1%8C-%D0%BD%D0%B0-%D1%87%D0%B5%D1%82%D1%8B%D1%80%D1%91%D1%85-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D0%B0%D1%85).

In [4]:
unit_path = path((1, 1, 1))
labels = unit_path.context.labels
indices = tuple(combinations(range(len(labels)), 2))
G = Matrix(green(unit_path).value.values)
R = Matrix(resistance(unit_path).value.values)
sums = [sum(R.row(i)) for i in range(4)]
grand_mean = sum(sums) / 16
gamma = zeros(4)
for i, j in indices:
    coefficient = (R[i, j] - sums[i] / 4 - sums[j] / 4 + grand_mean) / 2
    gamma[i, j] = gamma[j, i] = coefficient
    assert coefficient == -G[i, j]
assert tuple(gamma[i, j] for i, j in indices) == tuple(Fraction(k, 8) for k in (-1, 3, 5, 1, 3, -1))
signed = signed_forest_toponome(gamma)
assert signed[(3,)] == Fraction(1, 16)
print("P₄ signed vector coefficients:", tuple(gamma[i, j] for i, j in indices))

P₄ signed vector coefficients: (-1/8, 3/8, 5/8, 1/8, 3/8, -1/8)


## Разрезное раскрытие по исходным лесам

Перебираем восемь лесов исходного пути, строим дуальные формы их разбиений и сравниваем сумму с экспонентой гриниана.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%80%D0%B0%D0%B7%D1%80%D0%B5%D0%B7%D1%8B-%D0%B8-%D0%BB%D0%B5%D1%81%D0%BD%D1%8B%D0%B5-%D0%B3%D1%80%D0%B5%D0%B9%D0%B4%D1%8B-%D1%8D%D0%BA%D1%81%D0%BF%D0%BE%D0%BD%D0%B5%D0%BD%D1%82%D1%8B-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0#%D0%BB%D0%B5%D1%81%D0%BD%D0%B0%D1%8F-%D1%84%D0%BE%D1%80%D0%BC%D1%83%D0%BB%D0%B0-%D0%B3%D1%80%D0%B5%D0%B9%D0%B4%D0%BE%D0%B2-%D1%8D%D0%BA%D1%81%D0%BF%D0%BE%D0%BD%D0%B5%D0%BD%D1%82%D1%8B-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0).

In [5]:
forests = Counter()
for k in range(4):
    for edges in combinations(unit_path.edges, k):
        blocks = network(labels, edges).components
        forests[blocks] += prod(weight for _, _, weight in edges)
cut_sum = sum(
    (coefficient * cut_partition_form(unit_path.context, blocks).value
     for blocks, coefficient in forests.items()),
    start=0 * partition_form(unit_path.context, ()).value,
)
unit_N = exponential(unit_path, greenian_form)
assert len(forests) == 8
assert equivalent(cut_sum, unit_N).value
blocks = (("a", "b"), ("c",), ("d",))
cut = cut_partition_form(unit_path.context, blocks).value
top = partition_form(unit_path.context, (labels,)).value
assert dual_coefficients(cut, top) == affine_coefficients(partition_form(unit_path.context, blocks).value)
print("P₄: all eight original forests give the same cut exponential")

P₄: all eight original forests give the same cut exponential


## Миноры всех грейдов

В базисе разностей относительно последней вершины сравниваем коэффициенты каждого грейда экспоненты с соответствующими минорами.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D0%B2%D0%BD%D0%B5%D1%88%D0%BD%D1%8F%D1%8F-%D1%8D%D0%BA%D1%81%D0%BF%D0%BE%D0%BD%D0%B5%D0%BD%D1%82%D0%B0-%D0%B8-%D0%BF%D1%80%D0%B5%D0%B4%D1%81%D1%82%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0#%D0%BA%D0%BE%D1%8D%D1%84%D1%84%D0%B8%D1%86%D0%B8%D0%B5%D0%BD%D1%82%D1%8B-%D0%B2%D1%8B%D1%81%D1%88%D0%B8%D1%85-%D0%B3%D1%80%D0%B5%D0%B9%D0%B4%D0%BE%D0%B2-%D1%8F%D0%B2%D0%BB%D1%8F%D1%8E%D1%82%D1%81%D1%8F-%D0%BC%D0%B8%D0%BD%D0%BE%D1%80%D0%B0%D0%BC%D0%B8).

In [6]:
for net in (unit_path, weighted):
    matrix = Matrix(green(net).value.values)[:-1, :-1]
    exp_green = exponential(net, greenian_form)
    for k in range(4):
        blades = tuple(combinations(range(3), k))
        minors = {(left, right): matrix.extract(left, right).det()
                  for left in blades for right in blades}
        assert affine_coefficients(grade_part(exp_green, k).value) == {
            key: value for key, value in minors.items() if value
        }
print("All affine minors checked for both four-vertex paths")

All affine minors checked for both four-vertex paths


## Присоединение вершины

Присоединяем новую вершину к взвешенному треугольнику. Сверяем три вклада по степени новой вершины, определитель и остовной коэффициент.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D0%BA%D0%BE%D1%8D%D1%84%D1%84%D0%B8%D1%86%D0%B8%D0%B5%D0%BD%D1%82%D1%8B-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0-%D0%B8-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D0%BD%D1%8B%D0%B5-%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D1%86%D0%B8%D0%B8#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80%D1%8B-%D0%BF%D1%80%D0%B8%D1%81%D0%BE%D0%B5%D0%B4%D0%B8%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F).

In [7]:
triangle = network(tuple("abc"), (("a", "b", 2), ("b", "c", 3), ("a", "c", 5)))
attachments = (7, 11, 13)
attached = network(tuple("abcp"), triangle.edges + tuple(
    (a, "p", weight) for a, weight in zip(triangle.context.labels, attachments)
))
base_tau = spanning_tree_weight(triangle).value
ta, tb, tc = attachments
terms = (base_tau * sum(attachments), 2*(ta + tb)*tc + 3*(tb + tc)*ta + 5*(ta + tc)*tb,
         ta * tb * tc)
determinant = Matrix(laplacian(attached).value.values)[:3, :3].det()
assert terms == (961, 2072, 1001)
assert sum(terms) == determinant == spanning_tree_weight(attached).value == 4034
print("Attachment contributions by degree:", terms, "| tau =", determinant)

Attachment contributions by degree: (Fraction(961, 1), 2072, 1001) | tau = 4034


## Полный граф и звезда

Для полного графа и звезды суммируем корневые коэффициенты знаковых лесов по грейдам и проверяем нормировку предельной формы.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%8D%D0%BA%D1%81%D0%BF%D0%BE%D0%BD%D0%B5%D0%BD%D1%82%D1%8B-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%BE%D0%B2-%D1%82%D0%B8%D0%BF%D0%BE%D0%B2%D1%8B%D1%85-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2#%D0%BF%D0%BE%D0%BB%D0%BD%D1%8B%D0%B9-%D0%B3%D1%80%D0%B0%D1%84).

In [8]:
complete = network(labels, tuple((a, b, 1) for a, b in combinations(labels, 2)))
star = network(labels, (("a", "b", 1), ("a", "c", 1), ("a", "d", 1)))
for net, coefficients in ((complete, (1, Fraction(3, 4), Fraction(3, 16), Fraction(1, 64))),
                          (star, (1, Fraction(9, 4), Fraction(3, 2), Fraction(1, 4)))):
    G = Matrix(green(net).value.values)
    gamma_matrix = zeros(4)
    for i, j in indices:
        gamma_matrix[i, j] = gamma_matrix[j, i] = -G[i, j]
    forests = vertex_toponome(signed_forest_toponome(gamma_matrix))
    sums = tuple(sum(value for kind, value in forests.items() if sum(kind) == k) for k in range(4))
    assert sums == coefficients
    assert spanning_coefficient(exponential(net, greenian_form)).value == coefficients[-1] / 4
print("K₄ and the star: vertex coefficients and top-form normalization agree")

K₄ and the star: vertex coefficients and top-form normalization agree


## Потенциалы разностей, разрезов и границ

Для разности, разреза, границы тройки и предельной формы вычисляем потенциалы экспоненты гриниана с одной нормировкой.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D0%BA%D0%B0-%D0%B8-%D0%BF%D0%BE%D1%82%D0%B5%D0%BD%D1%86%D0%B8%D0%B0%D0%BB%D1%8B-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80%D1%8B-%D0%B8-%D0%BF%D1%80%D0%B5%D0%B4%D0%B5%D0%BB%D1%8C%D0%BD%D0%B0%D1%8F-%D1%84%D0%BE%D1%80%D0%BC%D0%B0).

In [9]:
forms = (
    partition_form(unit_path.context, (("a", "b"),)).value,
    cut_partition_form(unit_path.context, (("a", "b"), ("c", "d"))).value,
    partition_form(unit_path.context, (("a", "b", "c"),)).value,
    top,
)
unit_sigma = spanning_coefficient(unit_N).value
normalized = tuple(potential(unit_N, form).value / unit_sigma for form in forms)
assert normalized == (5, 1, 14, 16)
print("P₄ Greenian potentials: difference, cut, triangle, top =", normalized)

P₄ Greenian potentials: difference, cut, triangle, top = (Fraction(5, 1), Fraction(1, 1), Fraction(14, 1), Fraction(16, 1))


## Тождество для разрезов

Сравниваем сумму квадратов трёх разрезов с суммой квадратов четырёх центрированных вершин. Затем сравниваем их дуальные образы.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%81%D0%B8%D0%BC%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D1%82%D0%BE%D0%B6%D0%B4%D0%B5%D1%81%D1%82%D0%B2%D0%B0-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0#%D1%81%D0%B8%D0%BC%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D1%81%D1%83%D0%BC%D0%BC%D1%8B-%D1%80%D0%B0%D0%B7%D1%80%D0%B5%D0%B7%D0%BE%D0%B2).

In [10]:
def cut_form(side):
    complement = tuple(vertex for vertex in labels if vertex not in side)
    return cut_partition_form(unit_path.context, (tuple(side), complement)).value


left = sum((cut_form(side) for side in ("ab", "ac", "ad")), start=0 * top)
right = sum((cut_form((vertex,)) for vertex in labels), start=0 * top)
assert equivalent(left, right).value
assert dual_coefficients(left, top) == dual_coefficients(right, top)
print("Three cuts equal the sum of four centered vertex squares")

Three cuts equal the sum of four centered vertex squares


## Топономы дуальной пары

Сверяем полный знаковый топоном пути, обращение вершинных коэффициентов и определитель гриниана.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%82%D0%BE%D0%BF%D0%BE%D0%BD%D0%BE%D0%BC%D1%8B-%D1%8D%D0%BA%D1%81%D0%BF%D0%BE%D0%BD%D0%B5%D0%BD%D1%82%D1%8B-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D0%B5-%D0%BF%D0%B0%D1%80%D1%8B#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80%D1%8B-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D1%85-%D0%BF%D0%B0%D1%80).

In [11]:
assert signed == {
    (): 1, (1,): Fraction(5, 4), (2,): Fraction(3, 16),
    (1, 1): Fraction(15, 64), (3,): Fraction(1, 16),
}
rooted = vertex_toponome(signed)
p_green = tuple(sum(v for kind, v in rooted.items() if sum(kind) == k) for k in range(4))
p_laplacian = laplacian_polynomial(unit_path)
assert p_green == tuple(value / p_laplacian[-1] for value in reversed(p_laplacian))
t = symbols("t")
assert expand((eye(4) + t * Matrix(green(unit_path).value.values)).det()) == sum(
    value * t**k for k, value in enumerate(p_green)
)
print("Dual P₄ vertex coefficients:", p_laplacian, "→", p_green)

Dual P₄ vertex coefficients:

 (Fraction(1, 1), Fraction(6, 1), Fraction(10, 1), Fraction(4, 1)) → (Fraction(1, 1), Fraction(5, 2), Fraction(3, 2), Fraction(1, 4))
